# FraudGuard — Credit Card Fraud Detection for Retail Banking
### An end-to-end ML case study on the ULB Credit Card dataset

---

### Executive Summary

Credit card fraud causes **>USD 30 billion in losses globally each year** (Nilson Report). The detection problem has three structural challenges that distinguish it from textbook classification:

1. **Extreme class imbalance** — frauds are rarer than 0.2% of transactions.
2. **Asymmetric costs** — missing a fraud is far costlier than blocking a good transaction, but blocking erodes customer trust.
3. **Regulatory pressure** — every blocked transaction must be *explainable* to the customer and the regulator (RBI's *Master Directions on Digital Payment Security*, 2021).

This notebook walks through a rigorous workflow: EDA → leak-free feature engineering → benchmark of four model families with cross-validation → Bayesian hyperparameter tuning → probability calibration → cost-sensitive threshold optimisation → operational analysis (Precision@K for the fraud-ops queue) → Pareto-optimal decision frontier → SHAP explainability → statistical model comparison.

| Section | Technique | Why it matters for banking |
|---|---|---|
| 1 | Class-imbalance EDA | Sets evaluation strategy |
| 2 | Leak-free pipeline | Auditability |
| 3 | Stratified + temporal split | Real production is temporal |
| 4 | PR-AUC over ROC-AUC | Honest metric under imbalance |
| 5 | Optuna tuning | Reproducible model selection |
| 6 | Isotonic calibration | Probabilities feed into cost rules |
| 7 | Cost-sensitive threshold | Aligns ML with P&L |
| 8 | Precision@K | Matches ops team's review capacity |
| 9 | Pareto frontier | Decision-maker chooses operating point |
| 10 | SHAP | Required for adverse-action notices |
| 11 | McNemar test | Statistical evidence of improvement |

**Dataset:** ULB Credit Card Fraud Detection — 284,807 European transactions over 2 days, 492 frauds (0.172%). Features V1–V28 are PCA components (anonymised for confidentiality), plus `Time` and `Amount`.


## 0. Environment Setup

We use scikit-learn for the pipeline, LightGBM as the headline model, Optuna for Bayesian hyperparameter search, and SHAP for explainability.

In [ ]:
# Install non-default packages if missing (uncomment on first run)
# !pip install lightgbm optuna shap imbalanced-learn cloudpickle -q

import warnings, json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.datasets import fetch_openml
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import (average_precision_score, roc_auc_score, f1_score,
                             precision_score, recall_score, confusion_matrix,
                             classification_report, precision_recall_curve,
                             roc_curve, brier_score_loss)

sns.set_theme(style='whitegrid', context='notebook')
warnings.filterwarnings('ignore')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print('Environment ready.')

## 1. Data Loading

The ULB Credit Card dataset is available via OpenML (id 1597). The cell below tries OpenML first, then falls back to a local CSV (`creditcard.csv` in the working directory) — convenient if you have the Kaggle download.

In [ ]:
def load_creditcard():
    # 1. Try OpenML (works out-of-the-box if internet is available)
    try:
        ds = fetch_openml(name='creditcard', version=1, as_frame=True, parser='auto')
        df = ds.frame.copy()
        df['Class'] = df['Class'].astype(int)
        print('Loaded from OpenML.')
        return df
    except Exception as e:
        print(f'OpenML load failed ({type(e).__name__}); trying local file...')
    # 2. Fallback — local CSV (Kaggle download)
    for p in ['creditcard.csv', '../data/creditcard.csv', 'data/creditcard.csv']:
        if Path(p).exists():
            print(f'Loaded from local file: {p}')
            return pd.read_csv(p)
    raise FileNotFoundError(
        'Could not load dataset. Download creditcard.csv from '
        'https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud and place it in the working directory.'
    )

df = load_creditcard()
print(f'Shape: {df.shape}')
print(f'Fraud rate: {df["Class"].mean():.4%}  ({df["Class"].sum()} frauds in {len(df):,} transactions)')
df.head()

## 2. Exploratory Data Analysis

### 2.1 The imbalance problem
A naive classifier that always predicts "Normal" achieves **99.83% accuracy** on this dataset — and catches **zero frauds**. This is the *accuracy paradox*. We will use **PR-AUC (Average Precision)** as our headline metric; it is far more informative than ROC-AUC under severe imbalance because it focuses on the positive (rare) class.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

# Class counts
counts = df['Class'].value_counts()
axes[0].bar(['Normal', 'Fraud'], counts.values, color=['#3498db', '#e74c3c'])
for i, v in enumerate(counts.values):
    axes[0].text(i, v, f'{v:,}', ha='center', va='bottom', fontsize=11)
axes[0].set_title(f'Imbalance ratio: 1 : {counts[0] // counts[1]}')
axes[0].set_ylabel('Count')
axes[0].set_yscale('log')

# Cumulative fraud over time
df_sorted = df.sort_values('Time').reset_index(drop=True)
df_sorted['cum_fraud'] = df_sorted['Class'].cumsum()
axes[1].plot(df_sorted['Time'] / 3600, df_sorted['cum_fraud'], color='#e74c3c', linewidth=2)
axes[1].set_xlabel('Hours since first transaction')
axes[1].set_ylabel('Cumulative fraud count')
axes[1].set_title('Fraud accrual is roughly linear in time — no obvious regime shifts')

plt.tight_layout(); plt.show()

### 2.2 Temporal patterns

Fraud rate varies by hour of day. We will use this in feature engineering, but with caution — the dataset only spans 2 days, so "hour-of-day" is a weak proxy for true temporal behaviour.

In [ ]:
df['Hour'] = (df['Time'] // 3600) % 24
fraud_rate_hour = df.groupby('Hour')['Class'].agg(['mean', 'sum', 'count']).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
sns.barplot(x='Hour', y='mean', data=fraud_rate_hour, ax=axes[0], color='#e74c3c')
axes[0].set_title('Fraud rate by hour of day')
axes[0].set_ylabel('P(fraud | hour)')
axes[0].axhline(df['Class'].mean(), color='black', linestyle='--', alpha=0.5, label='Overall rate')
axes[0].legend()

sns.lineplot(x='Hour', y='count', data=fraud_rate_hour, ax=axes[1], marker='o', color='#3498db')
axes[1].set_title('Transaction volume by hour')
axes[1].set_ylabel('Count')
plt.tight_layout(); plt.show()

print('Peak fraud-rate hours:')
print(fraud_rate_hour.nlargest(5, 'mean')[['Hour', 'mean', 'sum', 'count']].to_string(index=False))

### 2.3 Transaction amount

Most fraud is on *small* amounts — fraudsters often probe with low-value transactions to test stolen cards (the "card-testing" attack). This is a known pattern in payment-card threat models.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

# Amount distribution
for label, color, name in [(0, '#3498db', 'Normal'), (1, '#e74c3c', 'Fraud')]:
    sns.kdeplot(np.log1p(df[df['Class']==label]['Amount']),
                ax=axes[0], label=name, color=color, fill=True, alpha=0.4)
axes[0].set_xlabel('log(1 + Amount)')
axes[0].set_title('Amount distribution by class')
axes[0].legend()

# Top discriminative PCA features (by mean diff)
fraud_means = df[df['Class']==1].iloc[:, 1:29].mean()
normal_means = df[df['Class']==0].iloc[:, 1:29].mean()
discriminative = (fraud_means - normal_means).abs().sort_values(ascending=False)
top4 = discriminative.head(4).index.tolist()
print(f'Top 4 discriminative PCA features: {top4}')

sample = df.sample(20000, random_state=RANDOM_STATE)
melted = sample.melt(id_vars='Class', value_vars=top4, var_name='Feature', value_name='Value')
sns.boxplot(x='Feature', y='Value', hue='Class', data=melted, ax=axes[1],
            palette=['#3498db', '#e74c3c'], showfliers=False)
axes[1].set_title('Distribution of top discriminative PCA features')
plt.tight_layout(); plt.show()

### 2.4 Feature relationship with Class

A full 30×30 heatmap is uninformative because V1–V28 are uncorrelated by construction (PCA outputs). The useful view is correlations *with the target*.

In [ ]:
target_corr = df.drop(columns=['Hour']).corr()['Class'].drop('Class').sort_values()
fig, ax = plt.subplots(figsize=(10, 7))
colors = ['#e74c3c' if v < 0 else '#2ecc71' for v in target_corr.values]
ax.barh(target_corr.index, target_corr.values, color=colors)
ax.set_xlabel('Pearson correlation with Class')
ax.set_title('Feature linear correlation with fraud label')
ax.axvline(0, color='black', linewidth=0.8)
plt.tight_layout(); plt.show()

## 3. Feature Engineering — leak-free

A subtle but critical bug in many fraud notebooks: features like *"transaction amount relative to hourly median"* are computed on the full dataframe **before** the train/test split. This leaks the test set's amount distribution into training.

We avoid this by wrapping engineering inside a `BaseEstimator + TransformerMixin` class. The transformer's `fit()` learns hourly medians **from training data only**, and `transform()` applies them to any input. Combined with `sklearn.pipeline.Pipeline`, leakage becomes mechanically impossible.

In [ ]:
class FraudFeatures(BaseEstimator, TransformerMixin):
    """Engineered features fitted on training data only.

    - log_amount: log(1+Amount) — handles right-skew
    - hour: hour-of-day from Time
    - V14*Amount, V4*Amount, V12*Amount: interaction terms with most predictive PCA components
    - amount_deviation_from_hourly_median: anomaly score relative to typical hourly spend
    """
    def __init__(self):
        self.hourly_median_ = None

    def fit(self, X, y=None):
        Xh = (X['Time'] // 3600) % 24
        tmp = pd.DataFrame({'hour': Xh, 'Amount': X['Amount']})
        self.hourly_median_ = tmp.groupby('hour')['Amount'].median()
        self.global_median_ = X['Amount'].median()
        return self

    def transform(self, X):
        X = X.copy()
        X['log_amount'] = np.log1p(X['Amount'])
        X['hour'] = (X['Time'] // 3600) % 24
        X['V14_x_Amount'] = X['V14'] * X['Amount']
        X['V4_x_Amount']  = X['V4']  * X['Amount']
        X['V12_x_Amount'] = X['V12'] * X['Amount']
        # Map hourly medians learned in fit, fall back to global median for unseen hours
        med = X['hour'].map(self.hourly_median_).fillna(self.global_median_)
        X['amount_deviation'] = X['Amount'] - med
        # Drop Time (raw seconds since first transaction has no production meaning)
        return X.drop(columns=['Time'])

# Quick sanity check on a small slice
fe = FraudFeatures()
sample_X = df.drop(columns=['Class', 'Hour']).head(1000)
sample_X_eng = fe.fit_transform(sample_X)
print('Engineered feature shape:', sample_X_eng.shape)
print('New columns:', [c for c in sample_X_eng.columns if c not in sample_X.columns])

## 4. Validation Strategy — stratified *and* temporal

We use two splits in parallel:

* **Stratified hold-out** (75/25) — preserves the fraud rate in both sets; primary metric reporting.
* **Temporal hold-out** — train on the first ~70% of `Time`, test on the last ~30%. Mimics production where the model is trained on the past and scored on the future. If the temporal-split metrics are noticeably worse than the stratified-split metrics, that's a *concept-drift* signal we must flag to the business.

For cross-validation during model selection, we use **5-fold stratified K-fold** on the training set only.

In [ ]:
X_all = df.drop(columns=['Class', 'Hour'])
y_all = df['Class']

# --- Stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.25, stratify=y_all, random_state=RANDOM_STATE
)

# --- Temporal split (sort by Time, take last 30% as test)
order = df['Time'].argsort().values
cut = int(0.7 * len(df))
train_idx, test_idx = order[:cut], order[cut:]
X_train_t, X_test_t = X_all.iloc[train_idx], X_all.iloc[test_idx]
y_train_t, y_test_t = y_all.iloc[train_idx], y_all.iloc[test_idx]

print(f'Stratified — train fraud rate: {y_train.mean():.4%}, test fraud rate: {y_test.mean():.4%}')
print(f'Temporal   — train fraud rate: {y_train_t.mean():.4%}, test fraud rate: {y_test_t.mean():.4%}')

## 5. Model Benchmarking with Cross-Validation

We benchmark four families:

| Model | Why include |
|---|---|
| Logistic Regression (balanced) | Linear baseline, interpretable, fast |
| Random Forest (balanced) | Captures non-linearities, robust |
| LightGBM (balanced) | Strong gradient-boosting baseline for tabular |
| Isolation Forest | Unsupervised — sanity check on whether labels add value |

All wrapped in a `Pipeline(FraudFeatures → StandardScaler → Model)`. Metrics reported are **mean PR-AUC across 5 stratified folds**, with standard deviation — single-split numbers are unreliable on this dataset.

In [ ]:
# LightGBM is optional; fall back gracefully if not installed
try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    HAS_LGB = False
    print('lightgbm not installed; skipping LightGBM benchmark. `pip install lightgbm` to enable.')

def build_pipe(model):
    return Pipeline([
        ('features', FraudFeatures()),
        ('scaler', StandardScaler()),
        ('clf', model),
    ])

candidates = {
    'LogReg': LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE),
    'RandomForest': RandomForestClassifier(class_weight='balanced', n_estimators=200,
                                           n_jobs=-1, random_state=RANDOM_STATE),
}
if HAS_LGB:
    candidates['LightGBM'] = lgb.LGBMClassifier(class_weight='balanced', n_estimators=300,
                                                learning_rate=0.05, num_leaves=31,
                                                random_state=RANDOM_STATE, verbose=-1)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_results = {}

for name, model in candidates.items():
    pipe = build_pipe(model)
    fold_scores = []
    t0 = time.time()
    for fold, (tr, va) in enumerate(skf.split(X_train, y_train)):
        pipe.fit(X_train.iloc[tr], y_train.iloc[tr])
        proba = pipe.predict_proba(X_train.iloc[va])[:, 1]
        fold_scores.append(average_precision_score(y_train.iloc[va], proba))
    cv_results[name] = {'mean': np.mean(fold_scores), 'std': np.std(fold_scores),
                        'folds': fold_scores, 'time_s': time.time() - t0}
    print(f'{name:14s}  PR-AUC = {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}'
          f'   ({time.time()-t0:.1f}s)')

cv_df = pd.DataFrame({k: [v['mean'], v['std']] for k, v in cv_results.items()},
                     index=['PR-AUC mean', 'PR-AUC std']).T
cv_df

### 5.1 Unsupervised baseline — Isolation Forest

Useful as a sanity check: if an *unsupervised* anomaly detector matches our supervised model, the labels aren't adding much value. In practice, supervised models with proper features dominate — but the IF score itself can be **stacked as a feature** into the supervised model (a common production trick).

In [ ]:
# Fit Isolation Forest on training set, predict anomaly score on validation slice
iso = IsolationForest(contamination=y_train.mean(), n_estimators=200,
                       random_state=RANDOM_STATE, n_jobs=-1)
# Feature-engineer first so IF sees the same features
fe_iso = FraudFeatures().fit(X_train)
iso.fit(fe_iso.transform(X_train))
iso_scores_train = -iso.score_samples(fe_iso.transform(X_train))  # higher = more anomalous
iso_scores_test  = -iso.score_samples(fe_iso.transform(X_test))

iso_pr_auc = average_precision_score(y_test, iso_scores_test)
print(f'Isolation Forest PR-AUC (test): {iso_pr_auc:.4f}')
print(f'For comparison, best supervised CV PR-AUC: {max(v["mean"] for v in cv_results.values()):.4f}')
print('=> Supervised labels add substantial signal.')

## 6. Hyperparameter Tuning with Optuna

Bayesian optimisation over the LightGBM hyperparameter space. We use a *modest* 20-trial budget here for runtime; for a production-grade study, 100–300 trials with multi-fold CV is appropriate. The objective is **mean PR-AUC across 3 folds** (3 not 5 to keep the inner loop fast).

In [ ]:
try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    HAS_OPTUNA = True
except ImportError:
    HAS_OPTUNA = False
    print('optuna not installed; using default LightGBM params. `pip install optuna` to enable.')

if HAS_OPTUNA and HAS_LGB:
    inner_skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

    def objective(trial):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 100, 500),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
            'num_leaves': trial.suggest_int('num_leaves', 15, 127),
            'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10, log=True),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'class_weight': 'balanced',
            'random_state': RANDOM_STATE,
            'verbose': -1,
            'n_jobs': -1,
        }
        scores = []
        for tr, va in inner_skf.split(X_train, y_train):
            pipe = build_pipe(lgb.LGBMClassifier(**params))
            pipe.fit(X_train.iloc[tr], y_train.iloc[tr])
            p = pipe.predict_proba(X_train.iloc[va])[:, 1]
            scores.append(average_precision_score(y_train.iloc[va], p))
        return np.mean(scores)

    study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
    study.optimize(objective, n_trials=20, show_progress_bar=False)

    best_params = study.best_params
    print(f'Best PR-AUC: {study.best_value:.4f}')
    print('Best params:')
    for k, v in best_params.items():
        print(f'  {k}: {v}')
else:
    best_params = {'n_estimators': 300, 'learning_rate': 0.05, 'num_leaves': 31,
                   'class_weight': 'balanced', 'random_state': RANDOM_STATE,
                   'verbose': -1, 'n_jobs': -1}
    print('Using default params.')

In [ ]:
# Visualise the tuning history
if HAS_OPTUNA and HAS_LGB:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    trial_values = [t.value for t in study.trials if t.value is not None]
    axes[0].plot(trial_values, marker='o', color='#3498db', alpha=0.7)
    axes[0].plot(np.maximum.accumulate(trial_values), color='#e74c3c', linewidth=2, label='Best so far')
    axes[0].set_xlabel('Trial'); axes[0].set_ylabel('PR-AUC')
    axes[0].set_title('Optuna tuning history'); axes[0].legend()

    # Parameter importances (approximate)
    try:
        imp = optuna.importance.get_param_importances(study)
        axes[1].barh(list(imp.keys()), list(imp.values()), color='#2ecc71')
        axes[1].set_title('Hyperparameter importance (fANOVA)')
        axes[1].set_xlabel('Importance')
    except Exception:
        axes[1].text(0.5, 0.5, 'Importance unavailable', ha='center', va='center')
    plt.tight_layout(); plt.show()

## 7. Probability Calibration

Tree ensembles output *scores* that resemble probabilities but are systematically biased — typically overconfident at the extremes. Downstream cost optimisation (Section 8) treats `predict_proba` as if it were a true probability, so miscalibration directly translates to a *wrong* operating threshold.

We wrap LightGBM in `CalibratedClassifierCV` with **isotonic** regression (preferred over Platt scaling when we have enough positives, as isotonic is non-parametric). The reliability diagram below shows the effect.

In [ ]:
if HAS_LGB:
    # Train uncalibrated LightGBM with best params
    base_model = lgb.LGBMClassifier(**best_params)
    pipe_uncal = build_pipe(base_model)
    pipe_uncal.fit(X_train, y_train)
    proba_uncal = pipe_uncal.predict_proba(X_test)[:, 1]

    # Calibrate via 3-fold isotonic
    calibrated = CalibratedClassifierCV(
        estimator=build_pipe(lgb.LGBMClassifier(**best_params)),
        method='isotonic', cv=3
    )
    calibrated.fit(X_train, y_train)
    proba_cal = calibrated.predict_proba(X_test)[:, 1]

    # Brier score: lower is better
    brier_uncal = brier_score_loss(y_test, proba_uncal)
    brier_cal   = brier_score_loss(y_test, proba_cal)
    pr_uncal    = average_precision_score(y_test, proba_uncal)
    pr_cal      = average_precision_score(y_test, proba_cal)

    print(f'Uncalibrated — PR-AUC: {pr_uncal:.4f}, Brier: {brier_uncal:.5f}')
    print(f'Calibrated   — PR-AUC: {pr_cal:.4f}, Brier: {brier_cal:.5f}')

    # Reliability diagram
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
    for proba, label, color in [(proba_uncal, 'Uncalibrated', '#e74c3c'),
                                 (proba_cal,   'Calibrated',   '#2ecc71')]:
        frac_pos, mean_pred = calibration_curve(y_test, proba, n_bins=10, strategy='quantile')
        axes[0].plot(mean_pred, frac_pos, marker='o', label=label, color=color)
    axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Perfect calibration')
    axes[0].set_xlabel('Mean predicted probability')
    axes[0].set_ylabel('Fraction of positives')
    axes[0].set_title('Reliability diagram')
    axes[0].legend()

    # Score distributions
    axes[1].hist(proba_cal[y_test==0], bins=50, alpha=0.5, label='Normal', color='#3498db', density=True)
    axes[1].hist(proba_cal[y_test==1], bins=50, alpha=0.5, label='Fraud', color='#e74c3c', density=True)
    axes[1].set_yscale('log')
    axes[1].set_xlabel('Calibrated fraud probability')
    axes[1].set_title('Score distribution by class (calibrated)')
    axes[1].legend()
    plt.tight_layout(); plt.show()

    final_proba = proba_cal
    final_model = calibrated
else:
    # Fallback to RandomForest if LightGBM is unavailable
    final_model = build_pipe(RandomForestClassifier(class_weight='balanced',
                              n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE))
    final_model.fit(X_train, y_train)
    final_proba = final_model.predict_proba(X_test)[:, 1]
    print(f'Using RandomForest fallback. PR-AUC: {average_precision_score(y_test, final_proba):.4f}')

## 8. Cost-Sensitive Decision Theory

The default `predict()` method uses threshold = 0.5. For an imbalanced cost-asymmetric problem, this is almost never optimal. We minimise **expected business cost** instead:

$$	ext{Cost}(t) = C_{FN} \cdot 	ext{FN}(t) + C_{FP} \cdot 	ext{FP}(t)$$

**Cost assumptions:** average fraud loss ≈ USD 100 (assumed; in practice, vary with transaction amount), customer-friction cost of a false block ≈ USD 10 (call-centre + relationship cost).

We then run a **sensitivity analysis** — how does the optimal threshold shift as the cost ratio changes? This tells the business *"if your cost estimates are off by 2×, the threshold moves from 0.30 to 0.25 — not catastrophic"*, or alternatively reveals fragility.

In [ ]:
C_FN, C_FP = 100, 10  # business-supplied costs in USD

thresholds = np.linspace(0.001, 0.999, 200)
costs, fns, fps, recalls, precisions = [], [], [], [], []

for t in thresholds:
    pred = (final_proba >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    costs.append(fn * C_FN + fp * C_FP)
    fns.append(fn); fps.append(fp)
    recalls.append(tp / (tp + fn) if (tp + fn) else 0)
    precisions.append(tp / (tp + fp) if (tp + fp) else 0)

optimal_idx = int(np.argmin(costs))
t_opt = thresholds[optimal_idx]

print(f'Optimal threshold: {t_opt:.3f}')
print(f'At t_opt — Recall: {recalls[optimal_idx]:.3f}, Precision: {precisions[optimal_idx]:.3f}')
print(f'Total cost: USD {costs[optimal_idx]:,.0f}  (FN={fns[optimal_idx]}, FP={fps[optimal_idx]})')
print(f'No-model cost (block nothing): USD {y_test.sum()*C_FN:,.0f}')
print(f'Cost reduction vs no-model:  {(1 - costs[optimal_idx] / (y_test.sum()*C_FN)) * 100:.1f}%')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

axes[0].plot(thresholds, costs, color='#e74c3c', linewidth=2)
axes[0].axvline(t_opt, color='#2ecc71', linestyle='--', label=f'Optimal t = {t_opt:.3f}')
axes[0].set_xlabel('Decision threshold'); axes[0].set_ylabel('Total cost (USD)')
axes[0].set_title('Cost curve')
axes[0].legend()

axes[1].plot(thresholds, precisions, label='Precision', color='#2ecc71')
axes[1].plot(thresholds, recalls,    label='Recall',    color='#e74c3c')
axes[1].axvline(t_opt, color='black', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Decision threshold')
axes[1].set_title('Precision–Recall vs threshold')
axes[1].legend()
plt.tight_layout(); plt.show()

### 8.1 Cost-ratio sensitivity

How robust is the optimal threshold to the cost assumptions?

In [ ]:
fn_grid = [25, 50, 100, 200, 500, 1000]
fp_grid = [1, 5, 10, 25, 50, 100]
sensitivity = np.zeros((len(fn_grid), len(fp_grid)))

for i, cfn in enumerate(fn_grid):
    for j, cfp in enumerate(fp_grid):
        costs_grid = [fns[k]*cfn + fps[k]*cfp for k in range(len(thresholds))]
        sensitivity[i, j] = thresholds[int(np.argmin(costs_grid))]

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(sensitivity, annot=True, fmt='.3f', cmap='RdYlGn_r',
            xticklabels=fp_grid, yticklabels=fn_grid, ax=ax,
            cbar_kws={'label': 'Optimal threshold'})
ax.set_xlabel('FP cost (USD per false block)')
ax.set_ylabel('FN cost (USD per missed fraud)')
ax.set_title('Optimal threshold under varying cost assumptions')
plt.tight_layout(); plt.show()

print('Interpretation: rows = miss-fraud cost, columns = false-alert cost.')
print('Higher FN:FP ratio => more aggressive (lower) threshold.')

## 9. Operational Analysis — Precision@K

In production, the fraud operations team can only manually review *K* alerts per day. If we send them too many low-quality alerts, **alert fatigue** sets in. The right metric is therefore **Precision@K**: of the top-K transactions ranked by fraud probability, what fraction are truly fraudulent?

This metric directly answers the business question: *"If we hire N analysts who can review K alerts/day, how many frauds will they actually catch?"*

In [ ]:
K_values = [50, 100, 200, 500, 1000, 2000]
order = np.argsort(-final_proba)  # descending
y_sorted = y_test.values[order]

precision_at_k, recall_at_k, frauds_caught = [], [], []
total_frauds = y_test.sum()
for K in K_values:
    topK = y_sorted[:K]
    precision_at_k.append(topK.sum() / K)
    recall_at_k.append(topK.sum() / total_frauds)
    frauds_caught.append(int(topK.sum()))

pk_df = pd.DataFrame({
    'K (alerts reviewed)': K_values,
    'Frauds caught': frauds_caught,
    'Total frauds in test': total_frauds,
    'Precision@K': [f'{p:.1%}' for p in precision_at_k],
    'Recall@K': [f'{r:.1%}' for r in recall_at_k],
})
print(pk_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(K_values, precision_at_k, marker='o', label='Precision@K', color='#2ecc71', linewidth=2)
ax.plot(K_values, recall_at_k, marker='s', label='Recall@K', color='#e74c3c', linewidth=2)
ax.set_xscale('log')
ax.set_xlabel('K (number of top-ranked alerts reviewed)')
ax.set_ylabel('Score')
ax.set_title('Operational scoring — Precision and Recall at K')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 10. Pareto Frontier — Multi-Objective Decision Frontier

Cost optimisation in Section 8 collapses two objectives (missed frauds, false alerts) into a single dollar number using point estimates of FN and FP costs. When those costs are uncertain, a more honest output is the **Pareto frontier**: the set of threshold choices for which you cannot reduce missed frauds without increasing false alerts, or vice versa.

The decision-maker — Head of Fraud Risk, in a real bank — picks an operating point on this curve based on their *current* risk appetite. This is standard practice in operations research for problems with conflicting objectives.

In [ ]:
# Each threshold produces a (FP, FN) point. The non-dominated front is the lower-left convex hull.
points = np.array(list(zip(fps, fns)))

def pareto_front(pts):
    """Return mask of Pareto-optimal (minimise both) points."""
    is_opt = np.ones(len(pts), dtype=bool)
    for i, p in enumerate(pts):
        if is_opt[i]:
            # A point dominates p if both coords are <= and at least one is strictly <
            is_opt[is_opt] = np.any(pts[is_opt] < p, axis=1)
            is_opt[i] = True
    return is_opt

mask = pareto_front(points)
pareto_pts = points[mask]
pareto_thresh = thresholds[mask]
# Sort by FP for nice plotting
order_p = np.argsort(pareto_pts[:, 0])
pareto_pts = pareto_pts[order_p]
pareto_thresh = pareto_thresh[order_p]

# Knee point — point on the frontier closest to the (0,0) ideal
norm_pts = pareto_pts / pareto_pts.max(axis=0)
dist = np.linalg.norm(norm_pts, axis=1)
knee = int(np.argmin(dist))

fig, ax = plt.subplots(figsize=(8.5, 5.5))
ax.scatter(points[:, 0], points[:, 1], s=8, alpha=0.3, color='gray', label='All thresholds')
ax.plot(pareto_pts[:, 0], pareto_pts[:, 1], '-o', color='#2ecc71', linewidth=2,
        markersize=5, label='Pareto frontier')
ax.scatter(pareto_pts[knee, 0], pareto_pts[knee, 1], s=200, color='#e74c3c',
           marker='*', zorder=5, label=f'Knee point (t={pareto_thresh[knee]:.3f})')
ax.set_xlabel('False positives (good transactions blocked)')
ax.set_ylabel('False negatives (frauds missed)')
ax.set_title('Pareto frontier of decision thresholds')
ax.set_xscale('symlog'); ax.set_yscale('symlog')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print(f'Knee point: threshold = {pareto_thresh[knee]:.3f}, '
      f'FP = {int(pareto_pts[knee, 0])}, FN = {int(pareto_pts[knee, 1])}')
print(f'Cost-optimal: threshold = {t_opt:.3f}, '
      f'FP = {fps[optimal_idx]}, FN = {fns[optimal_idx]}')
print('The two often differ — knee point ignores costs, cost-optimal uses them.')

### 10.1 Risk view — Expected Loss and Value-at-Risk

A risk-engineering framing of the same numbers. We treat the residual losses (frauds the model misses) as a random variable and report:

- **Expected Loss (EL)** = mean residual loss across hold-out.
- **Value-at-Risk (VaR₉₅)** = 95th percentile of *daily* residual loss — the loss that won't be exceeded with 95% confidence.

These are the same constructs used in credit and operational risk regulatory capital calculations.

In [ ]:
# At t_opt, identify missed frauds and their dollar amounts
pred_opt = (final_proba >= t_opt).astype(int)
missed_mask = (y_test.values == 1) & (pred_opt == 0)
missed_amounts = X_test['Amount'].values[missed_mask]

# Group by day (the 2-day dataset has ~48 hours; bucket by 24h windows in the test set's Time)
test_time = X_test['Time'].values
day_bucket = (test_time // (24 * 3600)).astype(int)
daily_loss = pd.Series(missed_amounts).groupby(pd.Series(day_bucket[missed_mask])).sum()

EL = missed_amounts.sum() / max(daily_loss.shape[0], 1)
VaR95 = np.percentile(missed_amounts, 95) if len(missed_amounts) else 0
CVaR95 = missed_amounts[missed_amounts >= VaR95].mean() if (missed_amounts >= VaR95).any() else 0

print(f'Residual fraud loss summary (at optimal threshold {t_opt:.3f}):')
print(f'  Frauds missed: {missed_mask.sum()} / {y_test.sum()} ({missed_mask.sum()/y_test.sum():.1%})')
print(f'  Total missed-fraud amount: USD {missed_amounts.sum():,.2f}')
print(f'  Expected daily loss (EL):  USD {EL:,.2f}')
print(f'  VaR95 (per-transaction):   USD {VaR95:,.2f}')
print(f'  CVaR95 (avg loss in tail): USD {CVaR95:,.2f}')

## 11. Explainability with SHAP

Banking regulators require that any *automated decision affecting a customer* be explainable. Under EU GDPR Article 22 and analogous RBI guidance, a customer whose transaction is blocked can demand a reason. SHAP gives us per-prediction feature attributions using Shapley values from cooperative game theory.

We use `TreeExplainer` (exact, fast for tree ensembles) on a sample of test transactions.

In [ ]:
try:
    import shap
    HAS_SHAP = True
except ImportError:
    HAS_SHAP = False
    print('shap not installed; skipping explainability. `pip install shap` to enable.')

if HAS_SHAP and HAS_LGB:
    # Re-fit a single uncalibrated LightGBM for clean SHAP (TreeExplainer wants a tree model, not the calibration wrapper)
    pipe_for_shap = build_pipe(lgb.LGBMClassifier(**best_params))
    pipe_for_shap.fit(X_train, y_train)

    # Apply the feature engineering to a sample of test
    fe_step = pipe_for_shap.named_steps['features']
    sc_step = pipe_for_shap.named_steps['scaler']
    sample_idx = np.random.RandomState(RANDOM_STATE).choice(len(X_test), 3000, replace=False)
    X_sample_eng = fe_step.transform(X_test.iloc[sample_idx])
    X_sample_scaled = sc_step.transform(X_sample_eng)

    explainer = shap.TreeExplainer(pipe_for_shap.named_steps['clf'])
    shap_values = explainer.shap_values(X_sample_scaled)
    # LightGBM binary -> shap_values is array of (n, n_features), positive class
    if isinstance(shap_values, list):
        shap_values = shap_values[1]

    plt.figure(figsize=(9, 6))
    shap.summary_plot(shap_values, X_sample_eng, feature_names=X_sample_eng.columns.tolist(),
                      show=False, max_display=15)
    plt.title('SHAP summary — feature impact on fraud probability', fontsize=11)
    plt.tight_layout(); plt.show()

In [ ]:
# Single-prediction explanation: pick a high-confidence fraud and show the contributing features.
if HAS_SHAP and HAS_LGB:
    proba_sample = pipe_for_shap.predict_proba(X_test.iloc[sample_idx])[:, 1]
    fraud_in_sample = np.where(y_test.iloc[sample_idx].values == 1)[0]
    if len(fraud_in_sample):
        # Pick the highest-confidence true fraud
        best = fraud_in_sample[np.argmax(proba_sample[fraud_in_sample])]
        contribs = pd.Series(shap_values[best], index=X_sample_eng.columns).sort_values(key=abs, ascending=False)

        fig, ax = plt.subplots(figsize=(9, 5))
        top = contribs.head(10)
        colors = ['#e74c3c' if v > 0 else '#3498db' for v in top.values]
        ax.barh(top.index[::-1], top.values[::-1], color=colors[::-1])
        ax.axvline(0, color='black', linewidth=0.6)
        ax.set_xlabel('SHAP value (impact on fraud-probability log-odds)')
        ax.set_title(f'Why this transaction was flagged — model score = {proba_sample[best]:.3f}')
        plt.tight_layout(); plt.show()

        print('Top contributors (positive = pushed toward FRAUD, negative = pushed toward NORMAL):')
        print(top.to_string())

## 12. Statistical Comparison — McNemar's Test

It is not enough to say *"LightGBM scored 0.83 PR-AUC and Logistic Regression scored 0.71"* — we need to show the difference is **statistically significant**, not an artefact of the particular train/test split. **McNemar's test** is the standard pairwise comparison for two classifiers on the same test set; it operates on the discordant pairs (cases where the two models disagree).

In [ ]:
from scipy.stats import chi2

# Fit LR and (best) LGBM on full train, predict on test
lr_pipe = build_pipe(LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE))
lr_pipe.fit(X_train, y_train)
lr_pred = lr_pipe.predict(X_test)

# Use the calibrated LGBM if present, else fallback
if HAS_LGB:
    lgb_pred = (final_proba >= 0.5).astype(int)
    rival_name = 'LightGBM (calibrated)'
else:
    lgb_pred = (final_proba >= 0.5).astype(int)
    rival_name = 'RandomForest'

# Build McNemar contingency
both_correct  = ((lr_pred == y_test) & (lgb_pred == y_test)).sum()
lr_only       = ((lr_pred == y_test) & (lgb_pred != y_test)).sum()
lgb_only      = ((lr_pred != y_test) & (lgb_pred == y_test)).sum()
both_wrong    = ((lr_pred != y_test) & (lgb_pred != y_test)).sum()

print('McNemar contingency table:')
print(pd.DataFrame([[both_correct, lgb_only], [lr_only, both_wrong]],
                   index=[f'{rival_name} correct', f'{rival_name} wrong'],
                   columns=['LR correct', 'LR wrong']).to_string())

# McNemar statistic with continuity correction
b, c = lr_only, lgb_only
stat = (abs(b - c) - 1)**2 / (b + c) if (b + c) > 0 else 0
pval = 1 - chi2.cdf(stat, df=1)
print(f'\nMcNemar statistic: {stat:.2f}, p-value: {pval:.2e}')
if pval < 0.05:
    winner = rival_name if c > b else 'LogReg'
    print(f'=> Reject H0 at α=0.05. {winner} is significantly better.')
else:
    print('=> Fail to reject H0; classifiers are statistically indistinguishable on this test set.')

## 13. Final Evaluation & Model Card

Final test-set metrics on the **stratified hold-out** and **temporal hold-out**. The gap between them is the model's *concept-drift risk* indicator — if temporal performance is materially worse, the model needs more frequent retraining.

In [ ]:
def eval_block(name, model, Xte, yte):
    proba = model.predict_proba(Xte)[:, 1]
    pred  = (proba >= t_opt).astype(int)
    tn, fp, fn, tp = confusion_matrix(yte, pred).ravel()
    return {
        'split': name,
        'PR-AUC': average_precision_score(yte, proba),
        'ROC-AUC': roc_auc_score(yte, proba),
        'Precision': precision_score(yte, pred),
        'Recall': recall_score(yte, pred),
        'F1': f1_score(yte, pred),
        'TP': tp, 'FP': fp, 'FN': fn, 'TN': tn,
        'Cost (USD)': fn*C_FN + fp*C_FP,
    }

# Stratified
metrics_strat = eval_block('Stratified', final_model, X_test, y_test)

# Temporal — refit on temporal training set
if HAS_LGB:
    temporal_model = CalibratedClassifierCV(
        estimator=build_pipe(lgb.LGBMClassifier(**best_params)), method='isotonic', cv=3
    )
else:
    temporal_model = build_pipe(RandomForestClassifier(class_weight='balanced',
                                  n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE))
temporal_model.fit(X_train_t, y_train_t)
metrics_temp = eval_block('Temporal', temporal_model, X_test_t, y_test_t)

final_table = pd.DataFrame([metrics_strat, metrics_temp]).set_index('split')
print('Final hold-out metrics (decision threshold = {:.3f}):'.format(t_opt))
print(final_table.round(4).to_string())

### 13.1 Model Card (one-page summary)

The model card is a standard artefact required by responsible-AI frameworks (Google, NIST). It is the document a model risk committee will actually read.

In [ ]:
model_card = {
    'name': 'FraudGuard-LightGBM-v1',
    'task': 'Binary classification (fraud vs normal)',
    'training_data': {
        'source': 'ULB Credit Card Fraud Detection',
        'rows': int(len(X_train)),
        'fraud_rate': float(y_train.mean()),
        'time_span': '2 days (Sept 2013, European cards)',
        'features': len(X_train.columns) + 6,  # +6 engineered
    },
    'model': 'LightGBM, class-weight balanced, isotonic-calibrated',
    'hyperparameters': best_params if HAS_LGB else 'RandomForest defaults',
    'evaluation': {
        'cv_strategy': 'StratifiedKFold(n=5)',
        'primary_metric': 'PR-AUC',
        'pr_auc_cv_mean': float(max(v['mean'] for v in cv_results.values())),
        'final_pr_auc_stratified': float(metrics_strat['PR-AUC']),
        'final_pr_auc_temporal': float(metrics_temp['PR-AUC']),
    },
    'operating_point': {
        'threshold': float(t_opt),
        'cost_assumptions_usd': {'C_FN': C_FN, 'C_FP': C_FP},
        'precision_at_threshold': float(metrics_strat['Precision']),
        'recall_at_threshold': float(metrics_strat['Recall']),
    },
    'intended_use': 'Real-time scoring of card-present and card-not-present transactions for fraud-ops review.',
    'limitations': [
        'Trained on 2-day European dataset; geographic and temporal generalisation untested.',
        'PCA features are anonymised — no domain-meaning attached to V1-V28.',
        'Card-testing fraud (small amounts) is over-represented; large-amount fraud sees fewer training examples.',
        'No reject-inference: declined transactions whose true label is unknown are not modelled.',
    ],
    'monitoring_plan': [
        'Weekly PSI on each feature vs training distribution.',
        'Monthly back-test of PR-AUC; trigger retrain if drop > 10%.',
        'Daily Precision@K on reviewed alerts; trigger investigation if < 30%.',
    ],
    'compliance_notes': [
        'SHAP-based reason codes available for adverse-action notices.',
        'Cost-sensitive threshold tuneable by Fraud Risk Head.',
        'Champion-challenger framework: this model is champion; LR is challenger.',
    ]
}
print(json.dumps(model_card, indent=2, default=str))

In [ ]:
# Persist the trained artefact
# cloudpickle serialises classes defined in __main__ (i.e. in this notebook)
# so the artifact can be loaded from a fresh Python process without redefining FraudFeatures.
# `pip install cloudpickle` if missing.
try:
    import cloudpickle as pickle
except ImportError:
    import pickle
artifact = {
    'model': final_model,
    'feature_engineer': None,  # already inside the pipeline
    'threshold': float(t_opt),
    'cost_assumptions': {'C_FN': C_FN, 'C_FP': C_FP},
    'model_card': model_card,
    'training_features': list(X_train.columns),
}
out_path = Path('fraudguard_model.pkl')
with open(out_path, 'wb') as f:
    pickle.dump(artifact, f)
print(f'Model artifact saved to: {out_path.resolve()}')

## 14. Conclusion & Interview Talking Points

### Summary of what this notebook demonstrates

| Capability | Where in the notebook | Why it matters in banking |
|---|---|---|
| EDA framed by class imbalance | §2 | Sets metric strategy (PR-AUC over accuracy) |
| Leak-free feature engineering via `sklearn` `Pipeline` | §3 | Auditable, reproducible — passes model validation |
| Stratified *and* temporal validation | §4 | Surfaces concept-drift risk |
| 5-fold CV with mean ± std reporting | §5 | Statistical reliability over a single split |
| Unsupervised baseline (Isolation Forest) | §5.1 | Quantifies the marginal value of labels |
| Bayesian hyperparameter tuning (Optuna) | §6 | Reproducible, principled model selection |
| Probability calibration (isotonic) | §7 | Required for cost-based decision making |
| Cost-sensitive threshold + sensitivity | §8 | Aligns ML output with P&L |
| Precision@K | §9 | Matches the fraud-ops team's review capacity |
| Pareto frontier | §10 | Multi-objective decision support |
| Expected Loss & VaR | §10.1 | Risk-engineering framing familiar to credit-risk teams |
| SHAP explainability | §11 | Adverse-action notice requirement |
| McNemar test | §12 | Statistical evidence of model improvement |
| Model card | §13.1 | Model risk management artefact |

### Likely interview questions, with crisp answers

> **"Why PR-AUC and not ROC-AUC?"**
> ROC-AUC is misleading under severe class imbalance because the False Positive Rate is dominated by the huge negative class. PR-AUC focuses on the positive class and falls below baseline ~0.5 only when the model is useless, which makes it discriminative across the realistic range.

> **"How do you prevent data leakage when computing the `amount_deviation` feature?"**
> The hourly median is fitted inside a `BaseEstimator` transformer whose `.fit()` only sees training data. The transformer is the first step of an `sklearn.Pipeline`, so cross-validation re-fits it inside each fold — no test data ever touches median computation.

> **"Why isotonic calibration rather than Platt scaling?"**
> Isotonic is non-parametric and handles arbitrary monotonic miscalibration shapes — gradient-boosting score distortion is rarely sigmoid-shaped. Platt is safer when positives are very scarce (<<100); here we have 492 frauds, enough for isotonic.

> **"How do you decide the decision threshold?"**
> Two ways, both in the notebook: cost-optimal (minimise C_FN·FN + C_FP·FP) when costs are well-defined, or Pareto frontier when they are not. In production, the Head of Fraud Risk picks a point on the Pareto curve; the threshold is a *business decision* informed by the model, not output by the model.

> **"How would you monitor this model in production?"**
> Three layers — input drift (PSI per feature, KS test on score distribution), performance drift (weekly Precision@K on reviewed alerts, monthly back-test PR-AUC), and outcome drift (chargeback rate, customer complaints from blocks). Model retrain trigger: any of {PSI > 0.25 on a top-5 feature, PR-AUC drop > 10%, Precision@K < 30%}.

> **"What's the biggest weakness of this approach?"**
> Three: (1) the PCA features remove the ability to do domain-driven feature engineering; (2) no reject-inference — declined transactions whose true label is unknown are excluded from training, biasing the model toward the *approved* distribution; (3) only 2 days of data limits any genuine temporal generalisation analysis.

### Stretch extensions (mention only if asked)

- Graph features on a card–merchant bipartite graph (PageRank, community detection) feeding LightGBM
- Sequential CUSUM or Bayesian-sequential detection over the per-card score stream
- Conformal prediction for calibrated *intervals* on the fraud probability
- Active learning loop to prioritise which low-confidence transactions to send for manual labelling
